# Section 4: Running the Agent Loop

*Duration: 25 minutes*

---

Section 3 defined three tools and demonstrated that tool selection depends on description quality. The tools exist as Python functions. The schemas exist as JSON definitions. But nothing is wired together yet.

This section connects the tools to the model. The agent loop sends a question along with tool definitions. The model decides which tool to call. The loop executes the tool, feeds the result back, and repeats until the model produces a final answer or exhausts its iteration budget.

By the end of this section, the same 10 evaluation questions from the Escalation Lab will have been run through the agent loop, and the results will be directly comparable to the passive RAG baseline.

## 4.1 Loading Tool Definitions

The tool definitions saved in Section 3 are loaded here. If you skipped Section 3, the pre-built file contains the same definitions.

In [6]:
import json

# Load tool definitions from Section 3
with open("../prebuilt/tool_definitions.json", "r", encoding="utf-8") as f:
    tool_data = json.load(f)

tool_definitions = tool_data["tool_definitions"]

print(f"Loaded {len(tool_definitions)} tool definitions:")
for td in tool_definitions:
    print(f"  - {td['function']['name']}")

Loaded 3 tool definitions:
  - rag_retrieval
  - calculator
  - no_answer


## 4.2 Connecting to the MaaS Endpoint

The same connection pattern from previous sections. The model, the endpoint, and the credentials do not change. Only the way we call the model changes: now we pass tool definitions alongside the messages.

In [7]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import config
from openai import OpenAI
from utils.retriever import Retriever

client = OpenAI(api_key=config.API_KEY, base_url=config.ENDPOINT_BASE)
collection = Retriever.load()

print(f"Connected to : {config.ENDPOINT_BASE}")
print(f"Model        : {config.MODEL_ID}")
print(f"Corpus       : {collection.count()} chunks")

Connected to : https://litellm-prod.apps.maas.redhatworkshops.io/v1
Model        : granite-3-2-8b-instruct
Corpus       : 30 chunks


## 4.3 A Single Tool Call: Anatomy of the Response

Before building the loop, look at what happens when we send a single question with tools attached. The model does not answer directly. It returns a `tool_calls` field in the response, a structured instruction telling the caller which function to execute and with what arguments.

This is the raw API contract. Understanding it is necessary before abstracting it into a loop.

In [8]:
# Send a single question with tools. Inspect the raw response structure
test_question = "What is the saving throw for a 3rd level Fighter against Dragon Breath?"

response = client.chat.completions.create(
    model=config.MODEL_ID,
    messages=[
        {
            "role": "system",
            "content": "You are a rules assistant for Basic Fantasy RPG. Use the available tools to answer questions about the game rules."
        },
        {"role": "user", "content": test_question}
    ],
    tools=tool_definitions,
    temperature=0.0
)

# Print the full response structure so participants can see how tool calls work
choice = response.choices[0]
print(f"Finish reason: {choice.finish_reason}")
print(f"Content      : {choice.message.content}")
print(f"Tool calls   : {len(choice.message.tool_calls) if choice.message.tool_calls else 0}")

if choice.message.tool_calls:
    for i, tc in enumerate(choice.message.tool_calls):
        print(f"\n  Tool call {i + 1}:")
        print(f"    ID       : {tc.id}")
        print(f"    Function : {tc.function.name}")
        print(f"    Arguments: {tc.function.arguments}")

print("\n--- Key observation ---")
print("The model did not answer the question. It returned a structured instruction")
print("telling us which tool to run and with what arguments. The agent loop's job")
print("is to execute that instruction and feed the result back.")

Finish reason: tool_calls
Content      : None
Tool calls   : 1

  Tool call 1:
    ID       : chatcmpl-tool-6d9c3b5c012e44108bda18690a584f64
    Function : rag_retrieval
    Arguments: {"query": "saving throw Fighter Dragon Breath"}

--- Key observation ---
The model did not answer the question. It returned a structured instruction
telling us which tool to run and with what arguments. The agent loop's job
is to execute that instruction and feed the result back.


> **Facilitator note:** Make sure participants see the `finish_reason` field. When the model wants to call a tool, `finish_reason` is `"tool_calls"` (or `"stop"` with tool_calls present, depending on the API). When it wants to answer directly, `finish_reason` is `"stop"` with content filled in. The loop checks this to know when to stop iterating.

## 4.4 Implementing the Tool Dispatcher

The agent loop needs a way to execute whichever tool the model selects. The dispatcher maps tool names to Python functions. When the model returns `{"name": "rag_retrieval", "arguments": {"query": "..."}}`, the dispatcher calls the corresponding function and returns the result.

In [9]:
import ast
import operator

# --- Tool implementations (from Section 3, using shared retriever) ---

def rag_retrieval(query: str) -> dict:
    """Search the Basic Fantasy RPG corpus for relevant chunks."""
    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=["documents", "distances"]
    )

    chunks = []
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        chunks.append({"text": doc, "distance": round(dist, 4)})

    return {"tool": "rag_retrieval", "query": query, "chunks": chunks}


_SAFE_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod,
    ast.Pow: operator.pow, ast.USub: operator.neg,
}

def _safe_eval_node(node):
    """Recursively evaluate an AST node using only safe arithmetic operations."""
    if isinstance(node, ast.Expression):
        return _safe_eval_node(node.body)
    elif isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    elif isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        left = _safe_eval_node(node.left)
        right = _safe_eval_node(node.right)
        return _SAFE_OPS[type(node.op)](left, right)
    elif isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval_node(node.operand))
    else:
        raise ValueError(f"Unsupported operation: {ast.dump(node)}")

def calculator(expression: str) -> dict:
    """Evaluate a mathematical expression safely using ast."""
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval_node(tree)
        return {"tool": "calculator", "expression": expression, "result": result}
    except (ValueError, SyntaxError, TypeError, ZeroDivisionError) as e:
        return {"tool": "calculator", "expression": expression, "error": str(e)}


def no_answer() -> dict:
    """Decline to answer when the corpus lacks sufficient information."""
    return {
        "answer": "I do not have enough information to answer this question from the available corpus.",
        "tool": "no_answer"
    }


# --- Dispatcher ---
TOOL_DISPATCH = {
    "rag_retrieval": rag_retrieval,
    "calculator": calculator,
    "no_answer": no_answer,
}


def execute_tool(name: str, arguments: dict) -> str:
    """
    Execute a tool by name with the given arguments.
    Returns a JSON string for the OpenAI API tool result format.
    """
    if name not in TOOL_DISPATCH:
        return json.dumps({"error": f"Unknown tool: {name}"})

    fn = TOOL_DISPATCH[name]
    result = fn(**arguments)
    return json.dumps(result, default=str)


print("Tool dispatcher ready.")
print(f"Registered tools: {list(TOOL_DISPATCH.keys())}")

Tool dispatcher ready.
Registered tools: ['rag_retrieval', 'calculator', 'no_answer']


## 4.5 The Agent Loop

The loop connects everything: the model, the tools, and the decision cycle. On each iteration:

1. Send the conversation history (including any prior tool results) to the model
2. If the model returns a tool call, execute it and append the result to the history
3. If the model returns a text answer, stop; that is the final answer
4. If the iteration budget is exhausted, stop with whatever the model last produced

The trace records every step. In production, this trace is the audit trail: which tools were called, what arguments were passed, what results came back, and what the model decided at each step.

In [10]:
def run_agent_loop(question, tools, client, model_id, max_iterations=3, verbose=True):
    """
    Run the agent loop: send question with tools, execute tool calls,
    feed results back, repeat until the model answers or budget is exhausted.

    Returns a trace dict with every step recorded for inspection.
    """
    system_prompt = (
        "You are a rules assistant for Basic Fantasy RPG. "
        "Use the available tools to answer questions about the game rules. "
        "If the retrieved context is sufficient, answer the question directly. "
        "If the context is insufficient or irrelevant, use the no_answer tool. "
        "If the question requires a calculation, use the calculator tool. "
        "Always base your answer on tool results, not prior knowledge."
    )

    # Initialize conversation with system prompt and user question
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    trace = {
        "question": question,
        "steps": [],
        "final_answer": None,
        "iterations": 0
    }

    for iteration in range(1, max_iterations + 1):
        trace["iterations"] = iteration

        if verbose:
            print(f"\n--- Iteration {iteration} ---")

        # Call the model with tool definitions
        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            tools=tools,
            temperature=0.0
        )

        choice = response.choices[0]

        # Check if the model wants to call a tool
        if choice.message.tool_calls:
            # Process each tool call (models may request multiple)
            # Append the assistant message with tool_calls to history first
            messages.append(choice.message)

            for tc in choice.message.tool_calls:
                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

                if verbose:
                    print(f"  Tool call: {tool_name}({json.dumps(tool_args)})")

                # Execute the tool
                tool_result = execute_tool(tool_name, tool_args)

                if verbose:
                    # Print a preview of the result (truncate long outputs)
                    preview = tool_result[:200] + "..." if len(tool_result) > 200 else tool_result
                    print(f"  Result:    {preview}")

                # Record this step in the trace
                trace["steps"].append({
                    "iteration": iteration,
                    "tool": tool_name,
                    "arguments": tool_args,
                    "result": json.loads(tool_result)
                })

                # Append the tool result to the conversation history
                # This is the format the API expects for tool responses
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": tool_result
                })
        else:
            # The model returned a text answer, so we are done
            final_answer = choice.message.content
            trace["final_answer"] = final_answer

            if verbose:
                print(f"  Final answer: {final_answer[:300]}")

            return trace

    # Budget exhausted. Make one final call without tools to force a text answer
    if verbose:
        print(f"\n--- Budget exhausted, forcing final answer ---")

    messages.append({
        "role": "user",
        "content": "Based on the tool results above, provide your final answer to the original question."
    })

    response = client.chat.completions.create(
        model=model_id,
        messages=messages,
        temperature=0.0
    )

    trace["final_answer"] = response.choices[0].message.content

    if verbose:
        print(f"  Final answer: {trace['final_answer'][:300]}")

    return trace


print("Agent loop defined.")

Agent loop defined.


## 4.6 Tracing a Single Question

Before running the full evaluation, trace one of the failing questions step by step. Watch which tool the model selects, what arguments it passes, and what it decides after seeing the tool result.

This is the question the passive pipeline got wrong in Section 1 because the retriever returned chunks that did not contain the saving throw table.

In [11]:
# Load the evaluation results to get the failing questions
with open("../prebuilt/eval_results.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

results = eval_data["results"]
failures = [r for r in results if r["classification"] != "pass"]

# Run one failing question through the agent loop with full trace output
test_failure = failures[0]
print(f"Question : {test_failure['question']}")
print(f"Expected : {test_failure['expected']}")

trace = run_agent_loop(
    question=test_failure["question"],
    tools=tool_definitions,
    client=client,
    model_id=config.MODEL_ID,
    max_iterations=3,
    verbose=True
)

print(f"\n{'=' * 60}")
print(f"Total iterations: {trace['iterations']}")
print(f"Tools called    : {[s['tool'] for s in trace['steps']]}")

Question : Why can't Elves roll higher than a d6 for hit points?
Expected : Elves use a d6 for hit points because that is the hit die assigned to the Elf combination class in Basic Fantasy RPG.

--- Iteration 1 ---
  Tool call: rag_retrieval({"query": "Elves hit points d6"})
  Result:    {"tool": "rag_retrieval", "query": "Elves hit points d6", "chunks": [{"text": "Elves are a combination class, incorporating the abilities of both the Fighter and the Magic-User. Because the Elf combin...

--- Iteration 2 ---
  Final answer: Elves use a d6 for hit points because they are a combination class, incorporating the abilities of both the Fighter and the Magic-User. The smaller hit die of the two component classes, a d6, is used by Elves, the same die used by Magic-Users. This is not a racial penalty but a design choice that re

Total iterations: 2
Tools called    : ['rag_retrieval']


> **Facilitator note:** Walk through the trace output line by line. The important observation is not just whether the final answer is correct; it is the *decision sequence*. Did the model select the right tool? Did it pass a good query? Did it use the tool result appropriately? Each of these is a separate quality dimension.

## 4.7 Running All 10 Questions

Run every evaluation question through the agent loop. The results will be compared against the passive RAG baseline in Section 5.

For questions that already passed in the passive pipeline, the agent loop should still produce correct answers; it should not regress. For the failing questions, the agent loop has the opportunity to recover by selecting different tools or retrying retrieval.

In [12]:
# Run all 10 evaluation questions through the agent loop
agent_loop_results = []

for r in results:
    print(f"\n{'=' * 70}")
    print(f"[{r['id']}] {r['question']}")
    print(f"Expected: {r['expected']}")
    print(f"Passive result: {r['classification']}")

    trace = run_agent_loop(
        question=r["question"],
        tools=tool_definitions,
        client=client,
        model_id=config.MODEL_ID,
        max_iterations=3,
        verbose=True
    )

    agent_loop_results.append({
        "id": r["id"],
        "question": r["question"],
        "expected": r["expected"],
        "category": r["category"],
        "passive_classification": r["classification"],
        "agent_answer": trace["final_answer"],
        "iterations": trace["iterations"],
        "tools_used": [s["tool"] for s in trace["steps"]],
        "trace": trace["steps"]
    })

print(f"\n{'=' * 70}")
print(f"Completed {len(agent_loop_results)} questions.")


[q01] What happens if a Thief fails an Open Locks attempt?
Expected: The Thief must wait until gaining another level of experience before trying again. It may only be tried once per lock.
Passive result: pass

--- Iteration 1 ---
  Tool call: rag_retrieval({"query": "Thief fails Open Locks attempt"})
  Result:    {"tool": "rag_retrieval", "query": "Thief fails Open Locks attempt", "chunks": [{"text": "Open Locks: A Thief may attempt to open a lock given a set of thieves' picks and tools. The base chance of suc...

--- Iteration 2 ---
  Final answer: If a Thief fails an Open Locks attempt, they cannot try that particular lock again until they have gained at least one additional level of experience. This is to reflect the meaningful consequences of failure in the Thief abilities. The Game Master may also rule that the failed attempt has triggered

[q02] Why can't Elves roll higher than a d6 for hit points?
Expected: Elves use a d6 for hit points because that is the hit die assigned to t

## 4.8 Scoring Agent Results

Use the same model-as-judge approach from the Escalation Lab to classify whether each agent answer is correct.

In [13]:
JUDGE_PROMPT = """You are an evaluation judge. Compare the EXPECTED answer to the ACTUAL answer.

The ACTUAL answer is correct if it conveys the same key facts as the EXPECTED answer,
even if the wording differs. Minor omissions of non-essential details are acceptable.

Respond with EXACTLY one JSON object:
{"classification": "pass" or "fail", "reason": "<one sentence>"}
"""

print(f"{'ID':<6} {'Passive':<12} {'Agent':<10} {'Iters':<7} {'Tools Used'}")
print("=" * 80)

for ar in agent_loop_results:
    # Score the agent's answer against the expected answer
    response = client.chat.completions.create(
        model=config.MODEL_ID,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": (
                f"QUESTION: {ar['question']}\n\n"
                f"EXPECTED: {ar['expected']}\n\n"
                f"ACTUAL: {ar['agent_answer']}"
            )}
        ],
        temperature=0.0
    )

    raw = response.choices[0].message.content.strip()
    try:
        judgment = json.loads(raw)
    except json.JSONDecodeError:
        judgment = {"classification": "error", "reason": "Could not parse judge response"}

    ar["agent_classification"] = judgment["classification"]
    ar["judge_reason"] = judgment.get("reason", "")

    passive_display = ar["passive_classification"] if ar["passive_classification"] == "pass" else "FAIL"
    agent_display = judgment["classification"] if judgment["classification"] == "pass" else "FAIL"
    tools_display = ", ".join(ar["tools_used"]) if ar["tools_used"] else "none"

    print(f"{ar['id']:<6} {passive_display:<12} {agent_display:<10} {ar['iterations']:<7} {tools_display}")

# Summary
passive_passes = sum(1 for ar in agent_loop_results if ar["passive_classification"] == "pass")
agent_passes = sum(1 for ar in agent_loop_results if ar["agent_classification"] == "pass")

print("=" * 80)
print(f"Passive RAG: {passive_passes}/10    Agent Loop: {agent_passes}/10")

ID     Passive      Agent      Iters   Tools Used
q01    pass         pass       2       rag_retrieval
q02    FAIL         pass       2       rag_retrieval
q03    FAIL         pass       2       rag_retrieval
q04    pass         pass       2       rag_retrieval
q05    pass         pass       2       rag_retrieval
q06    pass         FAIL       2       calculator
q07    pass         pass       2       rag_retrieval
q08    pass         pass       2       rag_retrieval
q09    pass         pass       2       rag_retrieval
q10    pass         pass       2       rag_retrieval
Passive RAG: 8/10    Agent Loop: 9/10


## 4.9 Saving Results

Save the full agent loop results (including traces) to the prebuilt directory. Section 5 will load these for evaluation.

In [14]:
output = {
    "metadata": {
        "generated_by": "04_Running_the_Agent_Loop.ipynb",
        "model": config.MODEL_ID,
        "max_iterations": 3,
        "tools": [td["function"]["name"] for td in tool_definitions],
        "total_questions": len(agent_loop_results),
        "agent_passes": sum(1 for ar in agent_loop_results if ar.get("agent_classification") == "pass")
    },
    "results": agent_loop_results
}

output_path = "../prebuilt/agent_loop_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, default=str)

print(f"Saved agent loop results to {output_path}")
print(f"  Questions : {len(agent_loop_results)}")
print(f"  Passes    : {output['metadata']['agent_passes']}/10")

Saved agent loop results to ../prebuilt/agent_loop_results.json
  Questions : 10
  Passes    : 9/10


> **FIELD TAKEAWAY**
>
> The agent loop is not magic. It is a while loop with a dispatch table. The model selects a tool, the loop executes it, the result goes back to the model, and the model decides what to do next. The trace records every decision. In production, this trace is how you debug failures, measure latency per tool, and identify which tool descriptions need improvement.

---

## What Comes Next

Section 5 evaluates the agent loop results on two dimensions: answer correctness and reasoning correctness. A correct answer reached through the wrong tool is not a reliable system. The evaluation framework in Section 5 separates these concerns.

Move to `05_Evaluation/05_Evaluation.ipynb`.